# Phase 6: Forecasting Baselines + LightGBM + XGBoost
This notebook runs the package-level Phase 6 pipeline end-to-end without duplicating business logic.

In [ ]:
from IPython.display import display

from retail_forecasting.evaluation import (
    BaselineBenchmarkConfig,
    ForecastEvaluationConfig,
    evaluate_forecast_models_pipeline,
    load_best_model_registry,
    load_forecast_metrics,
    run_baseline_benchmark_pipeline,
)
from retail_forecasting.forecasting_models import (
    ForecastModelTrainingConfig,
    load_forecasting_data_bundle,
    prepare_forecasting_matrices,
    train_forecast_models_pipeline,
)

In [ ]:
bundle = load_forecasting_data_bundle(use_llm_features=True)
matrices = prepare_forecasting_matrices(bundle)

print('Training source:', bundle.training_source)
print('Train rows:', len(bundle.train_frame))
print('Validation rows:', len(bundle.validation_frame))
print('Test rows:', len(bundle.test_frame))
print('Feature count:', len(matrices.feature_columns))
print('LLM merged columns:', bundle.llm_added_columns)
print('LLM features used in matrix:', matrices.llm_feature_columns_in_matrix)

In [ ]:
baseline_outputs = run_baseline_benchmark_pipeline(
    BaselineBenchmarkConfig(use_llm_features=True)
)
baseline_outputs

In [ ]:
train_outputs = train_forecast_models_pipeline(
    ForecastModelTrainingConfig(
        model='all',
        segment_mode='global',
        use_llm_features=True,
        use_tuned_params=True,
        random_state=42,
    )
)
train_outputs

In [ ]:
evaluation_outputs = evaluate_forecast_models_pipeline(
    ForecastEvaluationConfig(
        model='all',
        optimize_metric='wmape',
        segment_mode='global',
        use_llm_features=True,
    )
)
evaluation_outputs

In [ ]:
# Optional: short local tuning run. Keep trials small for quick checks.
# tune_outputs = tune_forecast_models_pipeline(
#     ForecastTuningConfig(model='all', optimize_metric='wmape', n_trials=10, random_state=42)
# )
# tune_outputs

In [ ]:
validation_metrics = load_forecast_metrics('validation')
test_metrics = load_forecast_metrics('test')
best_registry = load_best_model_registry()

display(validation_metrics.sort_values('wmape').head(10))
display(test_metrics.sort_values('wmape').head(10))
display(best_registry.head(20))